# Privacy Guardian — Colab GPU Worker + Ollama AI

Optional temporary compute runtime for **approved heavy work only**
(Ollama reasoning, vision/OCR parsing, embeddings). The laptop remains the
**system of record**. Colab is disposable.

**Two tunnel endpoints:**
1. **Job Dispatcher** — laptop sends jobs here, Colab processes them
2. **Ollama AI** — laptop routes AI inference to Colab's GPU Ollama server

## 1 · Configuration

In [ ]:
# ===== CONFIGURATION =====
REPO_URL = "https://github.com/S-Q-Ali/Mis-Clear.git"
COLAB_JOB_DISPATCHER_URL = ""  # e.g. https://xxx.trycloudflare.com/api
MAX_ITERATIONS = 30
POLL_INTERVAL_S = 5
PROBE_GPU = True
RUN_DIR = "/content/privacy-guardian"
OLLAMA_MODELS = ["qwen3:8b", "gemma3:4b"]

## 2 · Install Ollama (AI Server)

In [ ]:
# === Install Ollama ===
import subprocess, os, time, urllib.request

print("Installing Ollama...")
subprocess.run(["sh", "-c", "curl -fsSL https://ollama.com/install.sh | sh"], check=True)

# Start server in background
_proc = subprocess.Popen(    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ, "OLLAMA_HOST": "0.0.0.0:11434"}
)
time.sleep(3)

# Verify
try:
    urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=5)
    print("Ollama server is live!")
except Exception as e:
    print(f"Still starting: {e}")

## 3 · Download AI Models (GPU Accelerated)

In [ ]:
# === Download Models ===
import httpx

for model in OLLAMA_MODELS:
    print(f"Downloading {model}...")
    try:
        with httpx.stream("POST", "http://127.0.0.1:11434/api/pull",
                      json={"name": model}, timeout=600) as resp:
            for line in resp.iter_lines():
                if line:
                    print(f"  {line[:80]}")
        print(f"{model} ready!")
    except Exception as e:
        print(f"Error: {e}")

# List models
resp = httpx.get("http://127.0.0.1:11434/api/tags", timeout=10)
print("Available:", [m["name"] for m in resp.json().get("models", [])])